In [ ]:
import random
import time
import heapq
from IPython.display import clear_output


# =========================================================
# PEAS - 8 PUZZLE MODEL BASED REFLEX AGENT
# =========================================================

PEAS = {
    "P - Performance": "Đưa bàn cờ về đúng trạng thái đích, số bước càng ít càng tốt",
    "E - Environment": "Bàn cờ 3x3 gồm các số 1 đến 8 và ô trống 0",
    "A - Actuators": "UP, DOWN, LEFT, RIGHT, STOP",
    "S - Sensors": "Biết trạng thái bàn cờ hiện tại, vị trí ô trống, action hợp lệ, biết đã đạt goal chưa"
}

SIZE = 3

GOAL_STATE = (1, 2, 3,
              4, 5, 6,
              7, 8, 0)

ACTIONS = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1)
}


# =========================================================
# E - ENVIRONMENT
# =========================================================

def create_environment(start_state):
    return {
        "state": tuple(start_state),
        "step": 0,
        "last_action": None
    }


def copy_environment(env):
    return {
        "state": tuple(env["state"]),
        "step": env["step"],
        "last_action": env["last_action"]
    }


def get_blank_position(state):
    return state.index(0)


def is_inside(row, col):
    return 0 <= row < SIZE and 0 <= col < SIZE


def get_legal_actions(state):
    blank = get_blank_position(state)
    row, col = divmod(blank, SIZE)
    legal_actions = []

    for action, (dr, dc) in ACTIONS.items():
        new_row = row + dr
        new_col = col + dc

        if is_inside(new_row, new_col):
            legal_actions.append(action)

    return legal_actions


def apply_action_to_state(state, action):
    if action not in ACTIONS:
        return state

    if action not in get_legal_actions(state):
        return state

    blank = get_blank_position(state)
    row, col = divmod(blank, SIZE)

    dr, dc = ACTIONS[action]
    new_row = row + dr
    new_col = col + dc
    new_blank = new_row * SIZE + new_col

    temp = list(state)
    temp[blank], temp[new_blank] = temp[new_blank], temp[blank]

    return tuple(temp)


def apply_action(env, action):
    if action == "STOP":
        env["last_action"] = action
        return

    old_state = env["state"]
    new_state = apply_action_to_state(old_state, action)

    env["state"] = new_state
    env["step"] += 1
    env["last_action"] = action


# =========================================================
# S - SENSORS
# =========================================================

def is_goal(state):
    return state == GOAL_STATE


def sensor(env):
    state = env["state"]
    blank = get_blank_position(state)

    return {
        "state": state,
        "blank_position": blank,
        "legal_actions": get_legal_actions(state),
        "is_goal": is_goal(state),
        "step": env["step"],
        "last_action": env["last_action"]
    }


# =========================================================
# P - PERFORMANCE
# =========================================================

def manhattan_distance(state):
    total = 0

    for index, value in enumerate(state):
        if value == 0:
            continue

        goal_index = GOAL_STATE.index(value)

        current_row, current_col = divmod(index, SIZE)
        goal_row, goal_col = divmod(goal_index, SIZE)

        total += abs(current_row - goal_row) + abs(current_col - goal_col)

    return total


def performance(env):
    if env["state"] == GOAL_STATE:
        return 100 - env["step"]

    return -manhattan_distance(env["state"]) - env["step"]


# =========================================================
# KIỂM TRA BÀN CỜ CÓ GIẢI ĐƯỢC KHÔNG
# =========================================================

def count_inversions(state):
    nums = [x for x in state if x != 0]
    count = 0

    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] > nums[j]:
                count += 1

    return count


def is_solvable(state):
    # Với 8 puzzle 3x3: số nghịch thế chẵn thì giải được
    return count_inversions(state) % 2 == 0


# =========================================================
# MODEL - MÔ HÌNH BÊN TRONG CỦA AGENT
# =========================================================

class AgentModel:
    def __init__(self):
        self.current_state = None
        self.blank_position = None
        self.legal_actions = []
        self.visited_states = set()
        self.plan = []
        self.goal_state = GOAL_STATE
        self.total_replan = 0

    def update_model(self, percept):
        self.current_state = percept["state"]
        self.blank_position = percept["blank_position"]
        self.legal_actions = percept["legal_actions"]
        self.visited_states.add(percept["state"])


# =========================================================
# SEARCH TRONG MODEL - AI ĐỂ TẠO PLAN
# =========================================================

def get_neighbors_with_action(state):
    result = []

    for action in get_legal_actions(state):
        next_state = apply_action_to_state(state, action)
        result.append((action, next_state))

    return result


def astar_search(start_state):
    if not is_solvable(start_state):
        return None

    heap = []
    start_h = manhattan_distance(start_state)
    heapq.heappush(heap, (start_h, 0, start_state, []))

    visited_cost = {}

    while heap:
        f, g, state, path = heapq.heappop(heap)

        if state == GOAL_STATE:
            return path

        if state in visited_cost and visited_cost[state] <= g:
            continue

        visited_cost[state] = g

        for action, next_state in get_neighbors_with_action(state):
            new_g = g + 1
            new_h = manhattan_distance(next_state)
            new_f = new_g + new_h

            new_path = path + [action]
            heapq.heappush(heap, (new_f, new_g, next_state, new_path))

    return None


# =========================================================
# MODEL BASED REFLEX AGENT
# =========================================================

class ModelBasedReflexAgent:
    def __init__(self):
        self.model = AgentModel()

    def rule_match(self, percept):
        """
        Agent chọn action dựa vào percept hiện tại + model bên trong.

        Đây là model based reflex agent vì:
        - Agent không chỉ nhìn percept hiện tại.
        - Agent lưu current_state, visited_states, plan trong model.
        - Agent dùng model để biết mình đang ở đâu và nên đi bước nào tiếp theo.
        """

        self.model.update_model(percept)

        # Rule 1: Nếu đã đạt goal thì dừng
        if percept["is_goal"]:
            return "STOP", "IF state hiện tại là GOAL THEN STOP"

        # Rule 2: Nếu puzzle không giải được thì dừng
        if not is_solvable(percept["state"]):
            return "STOP", "IF puzzle không giải được THEN STOP"

        # Rule 3: Nếu chưa có plan thì dùng model để tạo plan bằng AI
        if len(self.model.plan) == 0:
            self.model.plan = astar_search(percept["state"])
            self.model.total_replan += 1

            if self.model.plan is None:
                return "STOP", "IF không tìm được plan THEN STOP"

            if len(self.model.plan) == 0:
                return "STOP", "IF plan rỗng THEN STOP"

            action = self.model.plan.pop(0)
            return action, "IF chưa có plan THEN dùng AI tạo plan, sau đó lấy action đầu tiên"

        # Rule 4: Nếu đã có plan thì lấy action tiếp theo trong plan
        action = self.model.plan.pop(0)

        if action in percept["legal_actions"]:
            return action, "IF đã có plan AND action hợp lệ THEN thực hiện action tiếp theo"

        # Rule 5: Nếu action trong plan không hợp lệ thì lập kế hoạch lại
        self.model.plan = astar_search(percept["state"])
        self.model.total_replan += 1

        if self.model.plan is None or len(self.model.plan) == 0:
            return "STOP", "IF plan lỗi và không lập lại được THEN STOP"

        action = self.model.plan.pop(0)
        return action, "IF action trong plan không hợp lệ THEN re-plan bằng A*"


# =========================================================
# TẠO BÀN CỜ NGẪU NHIÊN
# =========================================================

def shuffle_state(shuffle_times=40):
    state = GOAL_STATE
    last_action = None

    opposite = {
        "UP": "DOWN",
        "DOWN": "UP",
        "LEFT": "RIGHT",
        "RIGHT": "LEFT"
    }

    for _ in range(shuffle_times):
        legal = get_legal_actions(state)

        if last_action is not None:
            legal = [a for a in legal if a != opposite[last_action]] or legal

        action = random.choice(legal)
        state = apply_action_to_state(state, action)
        last_action = action

    return state


def input_state():
    while True:
        raw = input("Nhập 9 số từ 0 đến 8, cách nhau bằng dấu cách: ")

        try:
            nums = tuple(int(x) for x in raw.split())
        except:
            print("Bạn nhập sai. Ví dụ đúng: 1 2 3 4 5 6 7 0 8")
            continue

        if len(nums) != 9:
            print("Phải nhập đúng 9 số.")
            continue

        if sorted(nums) != list(range(9)):
            print("Phải có đủ các số từ 0 đến 8, không trùng nhau.")
            continue

        if not is_solvable(nums):
            print("Bàn cờ này không giải được. Nhập bàn khác nhé.")
            continue

        return nums


# =========================================================
# OUTPUT
# =========================================================

def line():
    print("=" * 82)


def print_peas():
    line()
    print("PEAS CHO 8 PUZZLE - MODEL BASED REFLEX AGENT")
    line()

    for key, value in PEAS.items():
        print(key + ":")
        print("  -", value)
        print()


def print_board(state):
    print("+------+------+------+")
    for r in range(SIZE):
        row_text = "|"
        for c in range(SIZE):
            value = state[r * SIZE + c]

            if value == 0:
                symbol = " "
            else:
                symbol = str(value)

            row_text += f"  {symbol:^2}  |"

        print(row_text)
        print("+------+------+------+")

    print("0 hoặc ô trống = blank tile")
    print()


def print_goal_state():
    line()
    print("GOAL STATE")
    line()
    print_board(GOAL_STATE)


def print_initial_state(initial_env):
    line()
    print("TRẠNG THÁI BAN ĐẦU")
    line()
    print("Step ban đầu        :", initial_env["step"])
    print("State ban đầu       :", initial_env["state"])
    print("Manhattan ban đầu   :", manhattan_distance(initial_env["state"]))
    print("Solvable            :", is_solvable(initial_env["state"]))
    print()
    print_board(initial_env["state"])


def print_current_state(env, percept, agent):
    line()
    print("TRẠNG THÁI HIỆN TẠI")
    line()
    print("Step hiện tại       :", env["step"])
    print("State hiện tại      :", percept["state"])
    print("Blank position      :", percept["blank_position"])
    print("Legal actions       :", percept["legal_actions"])
    print("Is goal             :", percept["is_goal"])
    print("Manhattan           :", manhattan_distance(percept["state"]))
    print("Performance         :", performance(env))
    print("Last action         :", percept["last_action"])
    print("Visited states      :", len(agent.model.visited_states))
    print("Plan còn lại        :", agent.model.plan)
    print()
    print_board(percept["state"])


def print_agent_decision(rule, action):
    line()
    print("MODEL BASED REFLEX AGENT DECISION")
    line()
    print("Rule   :", rule)
    print("Action :", action)
    print()


def print_history(history):
    line()
    print("LỊCH SỬ ACTION")
    line()

    if len(history) == 0:
        print("Chưa có action nào.")
    else:
        for item in history:
            print(item)

    print()


def print_result(env, agent):
    line()
    print("KẾT QUẢ")
    line()

    if env["state"] == GOAL_STATE:
        print("Trạng thái: ĐÃ GIẢI XONG 8 PUZZLE")
    else:
        print("Trạng thái: CHƯA GIẢI XONG")

    print("Tổng số bước        :", env["step"])
    print("Manhattan cuối      :", manhattan_distance(env["state"]))
    print("Performance cuối    :", performance(env))
    print("Visited states      :", len(agent.model.visited_states))
    print("Số lần lập plan     :", agent.model.total_replan)
    line()


# =========================================================
# RUN - AGENT TỰ GIẢI
# =========================================================

def run_model_based_agent(env, max_steps=100, delay=0.3, animate=True):
    agent = ModelBasedReflexAgent()
    initial_env = copy_environment(env)
    history = []

    while env["step"] < max_steps:
        if animate:
            clear_output(wait=True)

        percept = sensor(env)
        action, rule = agent.rule_match(percept)

        print_peas()
        print_goal_state()
        print_initial_state(initial_env)
        print_current_state(env, percept, agent)
        print_agent_decision(rule, action)
        print_history(history)

        if action == "STOP":
            print_result(env, agent)
            break

        old_state = env["state"]
        old_h = manhattan_distance(old_state)

        apply_action(env, action)

        new_state = env["state"]
        new_h = manhattan_distance(new_state)

        history.append(
            f"Step {env['step']:02d} | Action: {action:<5} | "
            f"Manhattan: {old_h} -> {new_h} | "
            f"State: {old_state} -> {new_state}"
        )

        if env["state"] == GOAL_STATE:
            if animate:
                clear_output(wait=True)

            percept = sensor(env)
            print_peas()
            print_goal_state()
            print_initial_state(initial_env)
            print_current_state(env, percept, agent)
            print_history(history)
            print_result(env, agent)
            break

        time.sleep(delay)

    if env["step"] >= max_steps and env["state"] != GOAL_STATE:
        print_result(env, agent)


# =========================================================
# RUN - MANUAL
# =========================================================

def run_manual(env):
    initial_env = copy_environment(env)
    history = []

    while True:
        clear_output(wait=True)
        percept = sensor(env)

        print_peas()
        print_goal_state()
        print_initial_state(initial_env)
        print_current_state(env, percept, ModelBasedReflexAgent())
        print_history(history)

        if percept["is_goal"]:
            print_result(env, ModelBasedReflexAgent())
            break

        print("Nhập action: UP / DOWN / LEFT / RIGHT / STOP")
        action = input("Action = ").upper().strip()

        if action == "STOP":
            print("Bạn đã dừng MANUAL mode.")
            print_result(env, ModelBasedReflexAgent())
            break

        if action not in ACTIONS:
            history.append(f"Action '{action}' không tồn tại.")
            continue

        if action not in percept["legal_actions"]:
            history.append(f"Action '{action}' không hợp lệ tại vị trí blank hiện tại.")
            continue

        old_state = env["state"]
        old_h = manhattan_distance(old_state)

        apply_action(env, action)

        new_state = env["state"]
        new_h = manhattan_distance(new_state)

        history.append(
            f"Step {env['step']:02d} | Action: {action:<5} | "
            f"Manhattan: {old_h} -> {new_h} | "
            f"State: {old_state} -> {new_state}"
        )


# =========================================================
# MAIN
# =========================================================

def main():
    clear_output(wait=True)

    print("CHƯƠNG TRÌNH 8 PUZZLE - MODEL BASED REFLEX AGENT")
    print()

    print("CHỌN TRẠNG THÁI BAN ĐẦU")
    print("1. Random bàn cờ")
    print("2. Tự nhập bàn cờ")
    choose_state = input("Nhập 1 hoặc 2: ").strip()

    if choose_state == "2":
        start_state = input_state()
    else:
        try:
            shuffle_times = int(input("Nhập số lần xáo trộn, ví dụ 20 hoặc 40: "))
        except:
            shuffle_times = 30

        if shuffle_times < 1:
            shuffle_times = 20

        start_state = shuffle_state(shuffle_times)

    env = create_environment(start_state)

    print()
    print("CHỌN CHẾ ĐỘ")
    print("1. MANUAL - Người chơi tự nhập action")
    print("2. AGENT - Model Based Reflex Agent tự giải")
    mode = input("Nhập 1 hoặc 2: ").strip()

    if mode == "1":
        run_manual(env)

    elif mode == "2":
        try:
            delay = float(input("Nhập delay giữa các bước, ví dụ 0.3: "))
        except:
            delay = 0.3

        run_model_based_agent(
            env=env,
            max_steps=100,
            delay=delay,
            animate=True
        )

    else:
        print("Bạn nhập sai chế độ.")


main()


PEAS CHO 8 PUZZLE - MODEL BASED REFLEX AGENT
P - Performance:
  - Đưa bàn cờ về đúng trạng thái đích, số bước càng ít càng tốt

E - Environment:
  - Bàn cờ 3x3 gồm các số 1 đến 8 và ô trống 0

A - Actuators:
  - UP, DOWN, LEFT, RIGHT, STOP

S - Sensors:
  - Biết trạng thái bàn cờ hiện tại, vị trí ô trống, action hợp lệ, biết đã đạt goal chưa

GOAL STATE
+------+------+------+
|  1   |  2   |  3   |
+------+------+------+
|  4   |  5   |  6   |
+------+------+------+
|  7   |  8   |      |
+------+------+------+
0 hoặc ô trống = blank tile

TRẠNG THÁI BAN ĐẦU
Step ban đầu        : 0
State ban đầu       : (0, 5, 2, 8, 1, 7, 4, 3, 6)
Manhattan ban đầu   : 14
Solvable            : True

+------+------+------+
|      |  5   |  2   |
+------+------+------+
|  8   |  1   |  7   |
+------+------+------+
|  4   |  3   |  6   |
+------+------+------+
0 hoặc ô trống = blank tile

TRẠNG THÁI HIỆN TẠI
Step hiện tại       : 20
State hiện tại      : (1, 2, 3, 4, 5, 6, 7, 8, 0)
Blank position      : 8